# 0.94645 from OOF files only — no public blend inside

@tilii7 said it plainly in the [complementary-models discussion](https://www.kaggle.com/competitions/playground-series-s6e9/discussion/740769):
re-blending public submission files fits the public board, and ensembling from OOF files is the only
thing that works with regularity. I have published both kinds of notebook here, so I owe a measurement of
the second kind.

Every member below ships out-of-fold predictions, and every weight is chosen on OOF. No public
submission file is blended in.

| member | what it is | OOF AUC | corr. with the tree ensemble |
|---|---|---|---|
| tree ensemble | six LightGBM / XGBoost pipelines that see the data differently ([library](https://www.kaggle.com/datasets/megayak/s6e9-six-feature-views-oof-library)) | 0.946345 | 1 |
| **RealMLP** | [@yekenot's PyTorch RealMLP](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch), unchanged, retrained on the same 10 folds, 3 seeds | 0.946182 | **0.99597** |
| XGBoost | [@najiama's triple-TE XGBoost](https://www.kaggle.com/code/najiama/xgboost-triple-te-dynamic-pruning-lb-0-94639), its own OOF | 0.946142 | 0.99815 |

| submission | public LB |
|---|---|
| tree ensemble alone (four views) | 0.94639 |
| **this stack, four boundary rules, one RealMLP seed** | **0.94644** |
| the same with three RealMLP seeds averaged | **0.94645** |

The OOF gain was +0.00006 and the board gave +0.00005. The RealMLP carries most of it: it is the
only strong member here that is not a tree, and at 0.99597 it is further from the trees than any tree
variant I built (the most different of those read 0.99761).

For scale, 0.94645 is where two weeks of public-file blending on this competition topped out. It is below
today's 0.94650, which is built on a leaderboard-probed anchor. The submission at the bottom of this
notebook is that 0.94650 blend with this stack inside it, so the score badge matches the top of the Code
tab; the analysis above is about the stack itself.

**The seed line is the part I did not expect.** Averaging three RealMLP seeds instead of one moved the
stack's OOF by +0.000001 and its public LB by +0.00001. An out-of-fold row is predicted by a single fold
model, a test row by the average of ten, so seed noise cancels on test in a way OOF cannot show. Choose
weights on OOF; average seeds for the submission.

In [ ]:
import glob, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

ROOT = "/kaggle/input"
ID, TARGET = "id", "Will_Buy_EV"

def find_one(*pats):
    for p in pats:
        h = sorted(glob.glob(f"{ROOT}/**/{p}", recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(" | ".join(pats))

rk = lambda v: rankdata(v, method="average") / len(v)

lib = pd.read_csv(find_one("oof_six_views.csv")).sort_values(ID).reset_index(drop=True)
lib_t = pd.read_csv(find_one("test_six_views.csv")).sort_values(ID).reset_index(drop=True)
g_oof = pd.read_csv(find_one("oof_realmlp_g.csv")).set_index(ID).reindex(lib[ID])["G_realmlp_3seed"].to_numpy()
g_test = pd.read_csv(find_one("test_realmlp_g.csv")).set_index(ID).reindex(lib_t[ID])["G_realmlp_3seed"].to_numpy()
x_oof = pd.read_csv(find_one("xgboost-triple-te-dynamic-pruning-lb-0-94639/oof_XGBOOST.csv", "oof_XGBOOST.csv")).set_index(ID).reindex(lib[ID]).iloc[:, 0].to_numpy()
x_test = pd.read_csv(find_one("xgboost-triple-te-dynamic-pruning-lb-0-94639/submission_XGBOOST.csv", "submission_XGBOOST.csv")).set_index(ID).reindex(lib_t[ID]).iloc[:, 0].to_numpy()
y = lib[TARGET].to_numpy(); fold = lib["fold"].to_numpy()
assert np.isfinite(g_oof).all() and np.isfinite(g_test).all() and np.isfinite(x_oof).all() and np.isfinite(x_test).all()

VIEWS = {"A_lgbm_triple_te_digits_3seed": .2, "B_xgb_on_A_features": .2, "C_no_digits_windows_lift_sm2_30_300": .1,
         "D_no_exact_key_ladder_windows": .2, "E_ladder25_250_2500_lift_sm5_50_500": .1, "F_exact_rate_as_init_score": .2}
trees_oof = rk(sum(w * rk(lib[c].to_numpy(dtype=float)) for c, w in VIEWS.items()))
trees_test = rk(sum(w * rk(lib_t[c].to_numpy(dtype=float)) for c, w in VIEWS.items()))

names = ["trees", "RealMLP", "XGBoost"]
M = np.stack([trees_oof, rk(g_oof), rk(x_oof)], 1)
Mt = np.stack([trees_test, rk(g_test), rk(x_test)], 1)
for i, n in enumerate(names):
    print(f"{n:<8} OOF {roc_auc_score(y, M[:, i]):.6f}   spearman vs trees {spearmanr(M[:, i], M[:, 0]).statistic:.5f}")

## Choosing the weights without fooling myself

In-sample weight tuning on OOF is the usual way a stack looks better than it is. So the weights are chosen
by greedy hill climbing (forward selection with replacement, seven steps) on nine folds and scored on the
tenth, ten times. The nested gain is the number that counts. Note where the weight goes: half to the six
trees together, and a quarter each to the two members that are not part of them.

In [ ]:
pos = y == 1

def auc_of(score, mask):
    s = score[mask]; yy = pos[mask]
    r = rankdata(s, method="average"); n1 = yy.sum(); n0 = len(yy) - n1
    return (r[yy].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def hill_climb(mask, steps=7):
    counts = np.zeros(M.shape[1], int)
    i0 = max(range(M.shape[1]), key=lambda i: auc_of(M[:, i], mask))
    counts[i0] = 1; cur = M[:, i0].copy(); cur_auc = auc_of(cur, mask)
    for _ in range(steps - 1):
        tot = counts.sum()
        a, i = max((auc_of((cur * tot + M[:, i]) / (tot + 1), mask), i) for i in range(M.shape[1]))
        if a <= cur_auc + 1e-9:
            break
        counts[i] += 1; cur = (cur * tot + M[:, i]) / (tot + 1); cur_auc = a
    return counts / counts.sum()

rows = []
for f in range(10):
    tr, va = fold != f, fold == f
    w = hill_climb(tr)
    rows.append([f, *np.round(w, 3), (auc_of(M @ w, va) - auc_of(M[:, 0], va)) * 1e5])
nested = pd.DataFrame(rows, columns=["fold", *names, "gain over trees x1e-5"])
print(f"nested gain: mean {nested.iloc[:, -1].mean():+.2f}e-5, positive in {(nested.iloc[:, -1] > 0).sum()}/10 folds")
nested

In [ ]:
W = hill_climb(np.ones(len(y), bool))
print("weights on all folds:", dict(zip(names, np.round(W, 4))))
assert np.isclose(W.sum(), 1) and W[1] > 0.1 and W[2] > 0.1, W   # the two non-tree members earn real weight
stack_oof, stack_test = rk(M @ W), rk(Mt @ W)
print(f"stack OOF {roc_auc_score(y, stack_oof):.6f}  vs trees {roc_auc_score(y, trees_oof):.6f}")

## Four boundary rules, checked on OOF first

These are the deterministic cells from [@taeyangg4's notebook](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master).
They are pure on this competition's 668,665 training rows (393/393 buyers, 0/1,257, 0/186, 0/7,157) and
fixed in advance, so no fitting happens inside the folds; the question is only what they add to a model
that already ranks those rows well.

In [ ]:
train = pd.read_csv(find_one("playground-series-s6e9/train.csv", "train.csv")).sort_values(ID).reset_index(drop=True)
test = pd.read_csv(find_one("playground-series-s6e9/test.csv", "test.csv")).sort_values(ID).reset_index(drop=True)
assert np.array_equal(train[ID].to_numpy(), lib[ID].to_numpy()) and np.array_equal(test[ID].to_numpy(), lib_t[ID].to_numpy())

def rule_shift(d):
    inc = d.Annual_Income_USD.to_numpy(float); km = d.Daily_Commute_km.to_numpy(float)
    env1 = d.Environmental_Concern_Level.to_numpy() == 1
    no_sub = d.Subsidy_Available.astype(str).to_numpy() == "No"
    anx_mh = d.Range_Anxiety_Level.isin(["Medium", "High"]).to_numpy()
    s = np.zeros(len(d))
    s[inc >= 170537] += 10; s[(inc >= 31004) & (inc <= 41970)] -= 10
    s[km >= 83] -= 5; s[(inc == 30000) & no_sub & (env1 | anx_mh)] -= 5
    return s

sh_tr, sh_te = rule_shift(train), rule_shift(test)
g = np.array([auc_of(stack_oof + sh_tr, fold == f) - auc_of(stack_oof, fold == f) for f in range(10)])
print(f"rules on the stack's OOF: {g.mean()*1e5:+.2f}e-5, positive in {(g > 0).sum()}/10 folds")
print(f"stack + rules OOF {roc_auc_score(y, stack_oof + sh_tr):.6f}")

## Submission

The stack is 0.00005 behind the public 0.94650, and diversity only pays near parity, so mixing it on top
of that file costs a digit (20% → 0.94649). Where it does fit is *inside* the recipe:
[@taeyangg4's blend](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master) puts
6% on a stack whose file is not public, and this one goes in that slot at the same weight and the same score.

In [ ]:
N = len(test)
anchor = pd.read_csv(find_one("submission_latest_best.csv")).sort_values(ID)
assert np.array_equal(anchor[ID].to_numpy(), lib_t[ID].to_numpy())
anchor = anchor[TARGET].to_numpy(dtype=float)

def lexrank(primary, secondary):
    o = np.lexsort((secondary, primary))
    r = np.empty(N); r[o] = np.arange(1, N + 1)
    return (r - 0.5) / N

blend = 0.90 * lexrank(anchor, g_test) + 0.06 * rk(stack_test) + 0.04 * rk(g_test)
final = lexrank(blend + sh_te, g_test)
assert len(np.unique(final)) == N and np.isfinite(final).all()
pd.DataFrame({ID: lib_t[ID].to_numpy(), TARGET: final}).to_csv("submission.csv", index=False)
print(f"wrote submission.csv | spearman(final, anchor) = {spearmanr(final, anchor).statistic:.6f}")
print(f"the stack on its own scores 0.94645; this file scores 0.94650")

## What this does and does not show

It shows that a stack built only from OOF-validated members reaches 0.94644 here, and that its
leaderboard gain matched its OOF gain almost exactly (+0.00005 against +0.00006). That agreement is the
property that matters on the private board.

It does not beat 0.94650. Mixed 20% into that file, this stack still costs 0.00001 — it is 0.00006 behind,
and diversity pays only near parity. Closing that gap takes another member at this strength that is not a
tree. RealMLP was the first; I have not found a second.

Two things I checked that did not help and you can skip: retraining RealMLP for four epochs instead of
two (best epoch stays 2), and feeding it income digit and quantisation categories (fold AUC identical to
five decimals — the exact-income embedding already holds them).

---
Credit: [@yekenot](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch) for RealMLP,
[@najiama](https://www.kaggle.com/code/najiama/xgboost-triple-te-dynamic-pruning-lb-0-94639) for the XGBoost and its OOF,
[@taeyangg4](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master) for the boundary rules,
@tilii7 for the criterion. The tree pipelines, the 10-fold RealMLP retrain and the measurements are mine.
If this saved you a stacking run, an upvote helps.